# Шаблон бэктеста QuantResearch

Шаблон Jupyter Notebook для исследования и запуска бэктестов в **Backtrader**. По умолчанию используется синтетический демонстрационный набор данных — подключение к брокеру и торговые операции не выполняются.

Запускать из окружения QuantResearch (`.venv`).

## 1. Импорт библиотек

In [ ]:
import sys
print(sys.executable)

In [ ]:
import backtrader as bt
import pandas as pd
import numpy as np

from FinamPy import FinamPy
from MOEXPy import MOEXPy
from FinLabPy.Core import Symbol

## 2. Параметры эксперимента

In [ ]:
INITIAL_CASH = 100_000.0
COMMISSION = 0.001
SMA_PERIOD = 20

## 3. Стратегия

Пример стратегии без выставления заявок. Добавьте торговую логику в метод `next`, когда будете готовы.

In [ ]:
class ResearchStrategy(bt.Strategy):
    params = (("sma_period", SMA_PERIOD),)

    def __init__(self):
        self.sma = bt.indicators.SimpleMovingAverage(
            self.data.close, period=self.p.sma_period
        )

    def next(self):
        # TODO: добавить условия входа/выхода и управление риском
        pass


## 4. Исторические данные

Ниже — **синтетические** данные для проверки шаблона. Позже замените этот блок загрузчиком исторических данных FinamPy/FinLabPy.

Backtrader ожидает DataFrame с индексом `DatetimeIndex` и столбцами `open`, `high`, `low`, `close`, `volume`.

In [ ]:
dates = pd.date_range("2025-01-01", periods=120, freq="B")
rng = np.random.default_rng(42)
close = 100 + np.cumsum(rng.normal(0, 1, len(dates)))
frame = pd.DataFrame(
    {
        "open": close,
        "high": close + 1,
        "low": close - 1,
        "close": close,
        "volume": 1000,
    },
    index=dates,
)
frame.index.name = "datetime"
frame.head()


## 5. Запуск бэктеста

In [ ]:
cerebro = bt.Cerebro()
cerebro.addstrategy(ResearchStrategy, sma_period=SMA_PERIOD)
cerebro.adddata(bt.feeds.PandasData(dataname=frame))
cerebro.broker.setcash(INITIAL_CASH)
cerebro.broker.setcommission(commission=COMMISSION)

start_value = cerebro.broker.getvalue()
results = cerebro.run()
end_value = cerebro.broker.getvalue()
print(f"Стартовый капитал: {start_value:,.2f}")
print(f"Конечный капитал: {end_value:,.2f}")


## 6. Результаты и заметки

На следующем этапе добавим метрики, сохранение результатов в SQLite и сравнение запусков. Пока можно записывать выводы эксперимента здесь.

**Гипотеза:** …

**Выводы:** …